In [1]:
import pandas as pd
import numpy as np

In [ ]:
# Edit distance with S, I, D
def wer_counts(ref_words, hyp_words):
    n = len(ref_words)
    m = len(hyp_words)

    dp = np.zeros((n + 1, m + 1), dtype=int)

    for i in range(n + 1):
        dp[i, 0] = i
    for j in range(m + 1):
        dp[0, j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i, j] = dp[i - 1, j - 1]
            else:
                dp[i, j] = min(
                    dp[i - 1, j - 1] + 1,  # substitution
                    dp[i, j - 1] + 1,      # insertion
                    dp[i - 1, j] + 1       # deletion
                )

    # Backtrace
    i, j = n, m
    S = I = D = 0

    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i - 1] == hyp_words[j - 1]:
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i, j] == dp[i - 1, j - 1] + 1:
            S += 1
            i -= 1
            j -= 1
        elif j > 0 and dp[i, j] == dp[i, j - 1] + 1:
            I += 1
            j -= 1
        else:
            D += 1
            i -= 1

    return S, I, D

In [ ]:
# Load ONLY required columns
pred_df = pd.read_csv(
    "text_extraction_final.csv",
    usecols=["ID", "TargetSurvey"]
)

gt_df = pd.read_csv(
    "text_extraction_manual_labels_final.csv",
    usecols=["ID", "TargetSurvey"]
)

# Keep only GT IDs
df = gt_df.merge(
    pred_df,
    on="ID",
    how="inner",
    suffixes=("_gt", "_pred")
)

In [ ]:
# Corpus-level WER
total_S = total_I = total_D = total_N = 0

for _, row in df.iterrows():
    ref = str(row["TargetSurvey_gt"]).strip().split()
    hyp = str(row["TargetSurvey_pred"]).strip().split()

    S, I, D = wer_counts(ref, hyp)

    total_S += S
    total_I += I
    total_D += D
    total_N += len(ref)

wer = (total_S + total_I + total_D) / total_N if total_N > 0 else 0.0

# Output
print(f"S = {total_S}")
print(f"I = {total_I}")
print(f"D = {total_D}")
print(f"N = {total_N}")
print(f"WER = {wer:.4f}")

S = 13
I = 10
D = 10
N = 532
WER = 0.0620
